### Instala modelos de tradução
Instala os pacotes de linguagem necessários para realizar a tradução de algumas colunas do dataset

In [45]:
from argostranslate import package
from tqdm import tqdm

package.update_package_index()

available_packages = package.get_available_packages()

package_to_install = next(
    filter(
        lambda x: x.from_code == "zh" and x.to_code == "en",
        available_packages
    )
)

download_path = package_to_install.download()
package.install_from_path(download_path)

### Importando as Bibliotecas
Importa as bibliotecas necessárias para a execução do código

In [46]:
from dotenv import load_dotenv
import mysql.connector
import pandas as pd
import os

### Carregando as Variáveis
Define variáveis globais de acesso ao banco de dados e aos arquivos

In [47]:
load_dotenv()

CSV_FILE           = 'ori.csv'
UNNORMALIZED_TABLE = 'raw_matches'

DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_USER     = os.getenv("DB_USER")
DB_NAME     = os.getenv("DB_NAME")

### Leitura do Dataset
Lê o dataset (.csv) e armazena em um dataframe pandas

In [48]:
df = pd.read_csv("ori.csv", encoding="gbk")

### Tradução de colunas
Realiza traduções necessárias

In [57]:
from argostranslate import translate
TRANSLATE = "matchType"

tqdm.pandas()

installed_languages = translate.get_installed_languages()

from_lang = next(lang for lang in installed_languages if lang.code == "zh")
to_lang   = next(lang for lang in installed_languages if lang.code == "en")

translator = from_lang.get_translation(to_lang)

valores_unicos = df[TRANSLATE].dropna().unique()

mapa_traducoes = {}

for texto in tqdm(valores_unicos):
    texto = str(texto)
    try:
        traducao = translator.translate(texto)
    except Exception as e:
        print(f"Erro: {e}")
        traducao = texto
    mapa_traducoes[texto] = traducao

df[TRANSLATE] = df[TRANSLATE].map(mapa_traducoes)


100%|██████████| 177/177 [00:06<00:00, 25.58it/s]


### Tradução de Tipos
Lê a informação de uma coluna do dataset e traduz para um tipo do MySQL

In [50]:
def mysql_type(dtype):
    if "int" in str(dtype):
        return "INT"
    elif "float" in str(dtype):
        return "FLOAT"
    elif "datetime" in str(dtype):
        return "DATETIME"
    else:
        return "TEXT"

###  String Preparatória para Query
Cria uma lista de strings que combina o nome de cada coluna com seu tipo SQL. <br>
```python
columns = ["coluna_1 INT", "coluna_2 FLOAT", ... , "coluna_n TEXT"]
```

In [51]:
columns = []

for col, dtype in df.dtypes.items():

    sql_type = mysql_type(dtype)

    columns.append(f"`{col}` {sql_type}")


`MatchID` INT
`matchType` TEXT
`gameset` INT
`MatchDate` TEXT
`Duration` TEXT
`win` TEXT
`Team1` TEXT
`Team1_region` TEXT
`Team1_Baron` INT
`Team1_Dra` INT
`Team1_Turts` INT
`Team1_ban1` TEXT
`Team1_ban2` TEXT
`Team1_ban3` TEXT
`Team1_ban4` TEXT
`Team1_ban5` TEXT
`Team2` TEXT
`Team2_region` TEXT
`Team2_Baron` INT
`Team2_Dra` INT
`Team2_Turts` INT
`Team2_ban1` TEXT
`Team2_ban2` TEXT
`Team2_ban3` TEXT
`Team2_ban4` TEXT
`Team2_ban5` TEXT
`Team1_player1_name` TEXT
`Team1_player1_pick` TEXT
`Team1_player1_K` INT
`Team1_player1_D` INT
`Team1_player1_A` INT
`Team1_player1_CS` INT
`Team1_player1_gold` INT
`Team1_player1_damage` FLOAT
`Team1_player1_tanking` FLOAT
`Team1_player2_name` TEXT
`Team1_player2_pick` TEXT
`Team1_player2_K` INT
`Team1_player2_D` INT
`Team1_player2_A` INT
`Team1_player2_CS` INT
`Team1_player2_gold` INT
`Team1_player2_damage` FLOAT
`Team1_player2_tanking` FLOAT
`Team1_player3_name` TEXT
`Team1_player3_pick` TEXT
`Team1_player3_K` INT
`Team1_player3_D` INT
`Team1_player3_

### String de Criação de Tabela
String contendo a query SQL que cria as tabelas referentes ao dataset (não normalizada)

In [52]:
create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {UNNORMALIZED_TABLE} (
        {", ".join(columns)}
    )
    CHARACTER SET utf8mb4;
"""

### Conexão Inicial para Criar Banco
Cria uma conexão para criar um banco de dados caso ele não exista

In [53]:
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    use_pure=True
)

cursor = conn.cursor()

cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")

cursor.close()
conn.close()

### Conexão para a criação das tabelas
Conecta no banco de dados para criar as tabelas

In [54]:
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    use_pure=True
)

cursor = conn.cursor()

cursor.execute(create_table_sql)

### Popula o banco de dados
Insere os valores nas tabelas do banco de dados

In [55]:
cols = ", ".join([f"`{c}`" for c in df.columns])
placeholders = ", ".join(["%s"] * len(df.columns))

insert_sql = f"""
    INSERT INTO {UNNORMALIZED_TABLE}
    ({cols})
    VALUES ({placeholders})
"""

values = [
    tuple(None if pd.isna(v) else v for v in row)
    for row in df.itertuples(index=False, name=None)
]

cursor.executemany(insert_sql, values)

conn.commit()
conn.close()